# WGAN-GP with AFHQ at 128x128

This notebook demonstrates a more modern GAN training setup than classic DCGAN while remaining practical on Apple Silicon.

Repo story:

* `archive/dcgan_celeba` = historical baseline for understanding basic GAN mechanics
* `wgan_gp_afhq` = stronger and more stable adversarial demo for local Mac training

Why this demo:

* It still clearly shows the generator and critic fighting it out during training.
* It is more stable and usually more effective than classic DCGAN.
* It is realistic to run locally, which makes it better for discussion, iteration, and teaching in Jupyter.

Notes:

* This notebook is intentionally notebook-first.
* Start small: 128x128 resolution, modest batch size, and a subset if needed.

## Introduction

Generative Adversarial Networks, or GANs, train two models at the same time:

* **Generator (G):** creates fake images from random noise
* **Critic (or Discriminator, D):** tries to distinguish real images from generated ones

The two models improve through competition. The generator gets better at producing realistic images, while the critic gets better at spotting flaws. This adversarial setup is what makes GANs interesting and also what makes them tricky to train.

### Why WGAN-GP

This notebook uses **WGAN-GP**: Wasserstein GAN with Gradient Penalty.

Why this choice:

* It is generally more stable than classic DCGAN training.
* It gives a more useful learning signal during training.
* It is less prone to some of the common failure modes of early GANs, such as unstable loss behavior and mode collapse.
* It is practical to run locally, which makes it a better fit for this notebook and for discussion in class.

### Why this replaces the archived DCGAN demo

The archived DCGAN notebook is still useful as a historical baseline. It shows the classic generator-versus-discriminator setup clearly, and it remains a good reference point for early GAN design.

This notebook replaces it as the main local adversarial demo because WGAN-GP is a stronger teaching example:

* more stable training
* better behavior on modern hardware
* clearer demonstration of adversarial learning without as much tutorial-era fragility

So the archived DCGAN remains the baseline.  
This notebook becomes the better practical demo.

## Environment and Hardware Check

This notebook is intended to run locally on Apple Silicon.

Recommended stack:

* Python 3.10, 3.11, or 3.12
* TensorFlow
* tensorflow-metal
* matplotlib
* pillow

Notes:

* This notebook was developed in an environment where Python 3.12 worked successfully.
* If you run into package compatibility issues, Python 3.11 is a good fallback.
* The goal is not to chase the most complex GAN implementation possible. The goal is to run a modern adversarial demo reliably on local hardware so the training dynamics are easy to discuss and inspect.

Before training, we should confirm:

* TensorFlow is installed
* Metal acceleration is available
* the GPU is visible to TensorFlow

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs visible to TensorFlow:", tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU acceleration is available.")
else:
    print("No GPU detected by TensorFlow. Training will run on CPU.")

In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
device_name = "GPU" if gpus else "CPU"

print("TensorFlow version:", tf.__version__)
print("Visible GPUs:", gpus)
print("Using device:", device_name)

### Paths and configs
In classic GANs, the second network is usually called the discriminator.<br>
In WGAN-GP, it is more accurately called the critic, because it scores images rather than outputting a real/fake probability.

In [ ]:

DATA_ROOT = Path("../../assets/datasets/afhq/data/train/dog")
SRC_DIR = Path("../../notebooks/wgan_gp_afhq").resolve()    
OUTPUT_ROOT = Path("../../results/wgan_gp_afhq/v2")

CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
SAMPLES_DIR = OUTPUT_ROOT / "samples"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

#Notes:
#* Batch size 8 is a safe starting point for MPS.
#* If memory allows, try 12 or 16.
#* If training is slow, drop to 64x64 first, then come back to 128x128.
config = {
    "image_size": 64,
    "batch_size": 16,
    "latent_dim": 256,
    "channels": 3,
    "epochs": 50,
    "lr_g": 1e-4,
    "lr_d": 5e-5,
    "beta_1": 0.0,
    "beta_2": 0.99,
    "critic_steps": 2,
    "gp_weight": 10.0,
    "sample_every": 250,
    "save_every_epoch": 1,
    "n_sample": 16,
}

config

In [ ]:
def count_images(root: Path):
    return len(list(root.rglob("*.jpg"))) + len(list(root.rglob("*.jpeg"))) + len(list(root.rglob("*.png")))

print("DATA_ROOT exists:", DATA_ROOT.exists())
if DATA_ROOT.exists():
    print("Image count:", count_images(DATA_ROOT))
    class_names = [p.name for p in DATA_ROOT.iterdir() if p.is_dir()]
    print("Classes found:", class_names)

## Data Loading and Preprocessing

Next we load the AFHQ training images, resize them to 128x128, and normalize pixel values to the range [-1, 1].

This matches the output range typically used by the generator when the final layer uses `tanh`.

In [ ]:
raw_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_ROOT,
    labels=None,
    image_size=(config["image_size"], config["image_size"]),
    batch_size=config["batch_size"],
    shuffle=True,
    seed=SEED,
)

print("Class names:", raw_dataset.class_names)

dataset = raw_dataset.map(lambda x: (tf.cast(x, tf.float32) / 127.5) - 1.0)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

print("Number of batches:", tf.data.experimental.cardinality(dataset).numpy())

In [ ]:
sample_batch = next(iter(dataset))

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    img = ((sample_batch[i] + 1.0) * 127.5).numpy().astype("uint8")
    ax.imshow(img)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Model definition - Generator and Critic
Next we define the two networks used in WGAN-GP:

* **Generator:** maps random latent vectors to synthetic images
* **Critic:** assigns scores to real and generated images

In classic GAN tutorials, the second network is usually called the discriminator.</br>
In WGAN-GP, it is more accurately called the critic because it produces a score rather than a real/fake probability.

In [ ]:
def build_generator(latent_dim, image_size=128, channels=3, base_channels=256):
    assert image_size in [32, 64, 128, 256], "Use a power-of-two image size."
    
    num_upsamples = int(np.log2(image_size)) - 2  # from 4x4 to target
    noise = keras.Input(shape=(latent_dim,), name="noise")

    x = layers.Dense(4 * 4 * base_channels, use_bias=False)(noise)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Reshape((4, 4, base_channels))(x)

    filters = base_channels
    for _ in range(num_upsamples):
        x = layers.UpSampling2D()(x)
        x = layers.Conv2D(filters, kernel_size=3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)
        filters = max(filters // 2, 32)

    output = layers.Conv2D(
        channels, kernel_size=3, padding="same", activation="tanh", name="generated_image"
    )(x)

    return keras.Model(noise, output, name="generator")

In [ ]:
def build_critic(image_size=128, channels=3, base_channels=64, max_channels=256):
    assert image_size in [32, 64, 128, 256], "Use a power-of-two image size."

    num_downsamples = int(np.log2(image_size)) - 2  # down to 4x4
    image = keras.Input(shape=(image_size, image_size, channels), name="image")

    x = image
    filters = base_channels

    for _ in range(num_downsamples):
        x = layers.Conv2D(filters, kernel_size=4, strides=2, padding="same")(x)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.Dropout(0.3)(x)
        filters = min(filters * 2, max_channels)

    x = layers.Flatten()(x)
    output = layers.Dense(1, name="critic_score")(x)

    return keras.Model(image, output, name="critic")

In [ ]:
# Instantiate models - generator and critic
generator = build_generator(
    latent_dim=config["latent_dim"],
    image_size=config["image_size"],
    channels=config["channels"],
)

critic = build_critic(
    image_size=config["image_size"],
    channels=config["channels"],
)

generator.summary()
critic.summary()

## WGAN-GP Definition - models, losses, optimizers

WGAN-GP changes both the loss function and the role of the second network.

Instead of predicting a real/fake probability, the critic assigns a score to each image.
Training then pushes the critic to score real images higher than generated ones, while the generator tries to raise the critic's scores on fake images.

A gradient penalty is added to keep the critic's gradients well-behaved, which improves stability compared with early GAN training.

In [ ]:
def generator_loss(fake_scores):
    return -tf.reduce_mean(fake_scores)


def critic_loss(real_scores, fake_scores):
    return tf.reduce_mean(fake_scores) - tf.reduce_mean(real_scores)

In [ ]:
def gradient_penalty(critic, real_images, fake_images):
    batch_size = tf.shape(real_images)[0]
    alpha = tf.random.uniform([batch_size, 1, 1, 1], 0.0, 1.0)
    interpolated = real_images + alpha * (fake_images - real_images)

    with tf.GradientTape() as gp_tape:
        gp_tape.watch(interpolated)
        pred = critic(interpolated, training=True)

    grads = gp_tape.gradient(pred, interpolated)
    grads = tf.reshape(grads, [batch_size, -1])
    grad_norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-12)
    gp = tf.reduce_mean((grad_norm - 1.0) ** 2)
    return gp

In [ ]:
generator_optimizer = keras.optimizers.Adam(
    learning_rate=config["lr_g"],
    beta_1=config["beta_1"],
    beta_2=config["beta_2"],
)

critic_optimizer = keras.optimizers.Adam(
    learning_rate=config["lr_d"],
    beta_1=config["beta_1"],
    beta_2=config["beta_2"],
)

In [ ]:
class WGAN_GP(keras.Model):
    def __init__(self, generator, critic, latent_dim, critic_steps=3, gp_weight=10.0):
        super().__init__()
        self.generator = generator
        self.critic = critic
        self.latent_dim = latent_dim
        self.critic_steps = critic_steps
        self.gp_weight = gp_weight

    def compile(self, g_optimizer, c_optimizer):
        super().compile()
        self.g_optimizer = g_optimizer
        self.c_optimizer = c_optimizer

    @tf.function
    def train_step(self, real_images):
        if isinstance(real_images, tuple):
            real_images = real_images[0]

        batch_size = tf.shape(real_images)[0]

        c_loss_total = 0.0
        gp_total = 0.0

        for _ in range(self.critic_steps):
            random_latent_vectors = tf.random.normal(
                shape=(batch_size, self.latent_dim)
            )

            with tf.GradientTape() as tape:
                fake_images = self.generator(random_latent_vectors, training=True)

                fake_scores = self.critic(fake_images, training=True)
                real_scores = self.critic(real_images, training=True)

                c_loss = critic_loss(real_scores, fake_scores)
                gp = gradient_penalty(self.critic, real_images, fake_images)
                total_c_loss = c_loss + self.gp_weight * gp

            c_grads = tape.gradient(total_c_loss, self.critic.trainable_variables)
            self.c_optimizer.apply_gradients(zip(c_grads, self.critic.trainable_variables))

            c_loss_total += c_loss
            gp_total += gp

        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))

        with tf.GradientTape() as tape:
            generated_images = self.generator(random_latent_vectors, training=True)
            gen_img_scores = self.critic(generated_images, training=True)
            g_loss = generator_loss(gen_img_scores)

        g_grads = tape.gradient(g_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(g_grads, self.generator.trainable_variables))

        return {
            "critic_loss": c_loss_total / tf.cast(self.critic_steps, tf.float32),
            "gp": gp_total / tf.cast(self.critic_steps, tf.float32),
            "generator_loss": g_loss,
        }

In [ ]:
wgan = WGAN_GP(
    generator=generator,
    critic=critic,
    latent_dim=config["latent_dim"],
    critic_steps=config["critic_steps"],
    gp_weight=config["gp_weight"],
)

wgan.compile(
    g_optimizer=generator_optimizer,
    c_optimizer=critic_optimizer,
)

## Train and Log Progress

Now we train the WGAN-GP model and record how the critic and generator evolve over time.

During training, we will:

* save generated sample grids at the end of each epoch
* optionally save model checkpoints
* track critic loss, generator loss, and gradient penalty

In [ ]:
# create fixed latent vectors for sampling during training
fixed_noise = tf.random.normal(
    shape=(config["n_sample"], config["latent_dim"]),
    seed=SEED
)

In [ ]:
# Create image-saving callback
class GANMonitor(keras.callbacks.Callback):
    def __init__(self, num_images, latent_dim, fixed_noise, samples_dir, checkpoint_dir=None, save_every_epoch=1, log_path=None):
        super().__init__()
        self.num_images = num_images
        self.latent_dim = latent_dim
        self.fixed_noise = fixed_noise
        self.samples_dir = Path(samples_dir)
        self.checkpoint_dir = Path(checkpoint_dir) if checkpoint_dir is not None else None
        self.save_every_epoch = save_every_epoch
        self.log_path = Path(log_path) if log_path is not None else None

        self.samples_dir.mkdir(parents=True, exist_ok=True)
        if self.checkpoint_dir is not None:
            self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        if self.log_path is not None:
            self.log_path.parent.mkdir(parents=True, exist_ok=True)
            if not self.log_path.exists():
                with open(self.log_path, "w") as f:
                    f.write("epoch,critic_loss,gp,generator_loss\n")

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        generated_images = self.model.generator(self.fixed_noise, training=False)
        generated_images = ((generated_images + 1.0) * 127.5)
        generated_images = tf.clip_by_value(generated_images, 0, 255)
        generated_images = tf.cast(generated_images, tf.uint8).numpy()

        ncols = 4
        nrows = int(np.ceil(self.num_images / ncols))

        fig, axes = plt.subplots(nrows, ncols, figsize=(8, 2 * nrows))
        axes = np.array(axes).reshape(-1)

        for i, ax in enumerate(axes):
            if i < self.num_images:
                ax.imshow(generated_images[i])
            ax.axis("off")

        plt.tight_layout()
        sample_path = self.samples_dir / f"epoch_{epoch + 1:03d}.png"
        plt.savefig(sample_path, bbox_inches="tight")
        plt.close(fig)

        c_loss = float(logs.get("critic_loss", np.nan))
        gp = float(logs.get("gp", np.nan))
        g_loss = float(logs.get("generator_loss", np.nan))

        print(
            f"Epoch {epoch + 1}: "
            f"critic_loss={c_loss:.4f}, "
            f"gp={gp:.4f}, "
            f"generator_loss={g_loss:.4f}"
        )

        if self.log_path is not None:
            with open(self.log_path, "a") as f:
                f.write(f"{epoch + 1},{c_loss},{gp},{g_loss}\n")

        if self.checkpoint_dir is not None and ((epoch + 1) % self.save_every_epoch == 0):
            gen_path = self.checkpoint_dir / f"generator_epoch_{epoch + 1:03d}.weights.h5"
            crit_path = self.checkpoint_dir / f"critic_epoch_{epoch + 1:03d}.weights.h5"

            self.model.generator.save_weights(gen_path)
            self.model.critic.save_weights(crit_path)

In [ ]:
# Instantiate callback
gan_monitor = GANMonitor(
    num_images=config["n_sample"],
    latent_dim=config["latent_dim"],
    fixed_noise=fixed_noise,
    samples_dir=SAMPLES_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    save_every_epoch=config["save_every_epoch"],
    log_path=OUTPUT_ROOT / "training_log.csv",
)

In [ ]:
# TRAIN
history = wgan.fit(
    dataset,
    epochs=config["epochs"],
    callbacks=[gan_monitor],
)

In [ ]:
# Inspect history keys
print(history.history.keys())

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(history.history["critic_loss"], label="critic_loss")
plt.plot(history.history["generator_loss"], label="generator_loss")
plt.plot(history.history["gp"], label="gradient_penalty")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.title("WGAN-GP Training Metrics")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Show latest saved sample grid
latest_sample = sorted(SAMPLES_DIR.glob("epoch_*.png"))[-1]
display(Image.open(latest_sample))

### Interpreting the logs

A few reminders:

* GAN losses do not behave like ordinary supervised-learning losses.
* Lower is not always better in a simple sense.
* The most important evidence is the generated image samples over time.
* The critic loss, generator loss, and gradient penalty are useful diagnostics, not a beauty contest scoreboard.

### Resume from Checkpoint

These cells let us reload saved generator and critic weights and continue training from a later point.

In [ ]:
from pathlib import Path
import re

def latest_checkpoint_epoch(checkpoint_dir: Path):
    gen_files = sorted(checkpoint_dir.glob("generator_epoch_*.weights.h5"))
    crit_files = sorted(checkpoint_dir.glob("critic_epoch_*.weights.h5"))

    if not gen_files or not crit_files:
        return None, None, None

    pattern = re.compile(r"epoch_(\d+)\.weights\.h5$")

    gen_map = {}
    for p in gen_files:
        m = pattern.search(p.name)
        if m:
            gen_map[int(m.group(1))] = p

    crit_map = {}
    for p in crit_files:
        m = pattern.search(p.name)
        if m:
            crit_map[int(m.group(1))] = p

    common_epochs = sorted(set(gen_map.keys()) & set(crit_map.keys()))
    if not common_epochs:
        return None, None, None

    last_epoch = common_epochs[-1]
    return last_epoch, gen_map[last_epoch], crit_map[last_epoch]

In [ ]:
last_epoch, gen_ckpt, crit_ckpt = latest_checkpoint_epoch(CHECKPOINT_DIR)

print("Last epoch found:", last_epoch)
print("Generator checkpoint:", gen_ckpt)
print("Critic checkpoint:", crit_ckpt)

In [ ]:
if gen_ckpt is not None and crit_ckpt is not None:
    generator.load_weights(gen_ckpt)
    critic.load_weights(crit_ckpt)
    print(f"Loaded generator and critic weights from epoch {last_epoch}.")
else:
    print("No matching checkpoints found.")

In [ ]:
resume_epochs = 5  # change as needed

history_resume = None
if gen_ckpt is not None and crit_ckpt is not None:
    history_resume = wgan.fit(
        dataset,
        initial_epoch=last_epoch,
        epochs=last_epoch + resume_epochs,
        callbacks=[gan_monitor],
    )

In [ ]:
def merge_histories(*histories):
    merged = {}
    for hist in histories:
        if hist is None:
            continue
        h = hist.history if hasattr(hist, "history") else hist
        for k, v in h.items():
            merged.setdefault(k, []).extend(v)
    return merged

In [ ]:
full_history = merge_histories(history, history_resume)
print(full_history.keys())

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(full_history["critic_loss"], label="critic_loss")
plt.plot(full_history["generator_loss"], label="generator_loss")
plt.plot(full_history["gp"], label="gradient_penalty")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.title("WGAN-GP Training Metrics")
plt.legend()
plt.grid(True)
plt.show()

### Training Animation

This turns the saved sample grids into a GIF so we can see the generator improve over time.

In [ ]:
import imageio.v2 as imageio

def build_gif(samples_dir: Path, output_path: Path, fps: int = 2):
    image_paths = sorted(samples_dir.glob("epoch_*.png"))
    if not image_paths:
        raise FileNotFoundError(f"No epoch images found in {samples_dir}")

    frames = [imageio.imread(p) for p in image_paths]
    imageio.mimsave(output_path, frames, fps=fps)
    return output_path

def build_mp4(samples_dir: Path, output_path: Path, fps: int = 2):
    image_paths = sorted(samples_dir.glob("epoch_*.png"))
    if not image_paths:
        raise FileNotFoundError(f"No epoch images found in {samples_dir}")

    writer = imageio.get_writer(output_path, fps=fps)
    try:
        for p in image_paths:
            writer.append_data(imageio.imread(p))
    finally:
        writer.close()

    return output_path

In [ ]:
gif_path = OUTPUT_ROOT / "training_progress.gif"
build_gif(SAMPLES_DIR, gif_path, fps=2)
print("Saved GIF to:", gif_path)

mp4_path = OUTPUT_ROOT / "training_progress.mp4"
build_mp4(SAMPLES_DIR, mp4_path, fps=2)
print("Saved MP4 to:", mp4_path)

In [ ]:
from IPython.display import Image as IPyImage, display

display(IPyImage(filename=str(gif_path)))

### Talking points

Why this is a better demo than archived DCGAN:
- stronger architecture
- better image quality
- more stable adversarial training
- still portable enough to run on common hardware

Why not official StyleGAN3 here:
- optimized for NVIDIA/CUDA stack
- poor fit for a MacBook-based live training demo

What students should notice:
- generator improves through competition with discriminator
- image quality rises gradually, not magically
- adversarial training is unstable enough to be interesting, but not so fragile that the demo collapses instantly

## Comparison to archived DCGAN

Questions to discuss:
- What looks better?
- What trains more stably?
- What changed architecturally?
- Which setup is more realistic for modern workflows on local hardware?

## Next steps

After the first working run:
- add FID or a lightweight proxy metric
- test 64x64 vs 128x128
- try AFHQ vs CelebA
- add EMA for generator weights
- compare package-based StyleGAN2 vs custom fallback implementation


# ARCHIVE

apexsystems@okta.com

Amman Aurora? -- SRE contact from 
Jonathan Connales -- (NBC Univ) Al Maier's right-hand man. 
Al Maier - the client -- (NBC Univ) runs the group.
Carissa Golmer -- project management 
Johnathan 
James Reid - PMO on our side. He'll lead the scrum stuff
Eric D Fillabaum - 
Corey Davis
Michael Aggawal - technical oversight - stepping down. Ammal (newbie) will be in our tech meetings tomorrow
Pierce Shagin - engagement manager - write correspondence - spread very thin, so not very involved w/ our project.
Regin Terretti - AI/ML counterpart at Apex.
Sergi Cheshenko - AWS architect - working on ML Ops, not familiar w/ that. He's dropping off in about a month. 

Look at terraform,
